In [1]:
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)

device: cpu


In [8]:
transform = transforms.ToTensor()

full_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

In [9]:
print("完整训练集大小：", len(full_dataset))

完整训练集大小： 60000


In [10]:
train_size = 2000
val_size = 1000
unused_size = len(full_dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset, unused_dataset = random_split(
    full_dataset,
    [train_size, val_size, unused_size],
    generator=generator
)

In [11]:
print("训练集：", len(train_dataset))
print("验证集：", len(val_dataset))
print("暂未使用：", len(unused_dataset))

训练集： 2000
验证集： 1000
暂未使用： 57000


In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False
)

In [14]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784,256),
    nn.ReLU(),
    nn.Linear(256,128),
    nn.ReLU(),
    nn.Linear(128,10)
)
model = model.to(device)
print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): ReLU()
  (5): Linear(in_features=128, out_features=10, bias=True)
)


In [15]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)


In [17]:
num_epochs = 20

baseline_train_loss_list = []
baseline_train_acc_list = []
baseline_val_loss_list = []
baseline_val_acc_list = []

for epoch in range(num_epochs):
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images,labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        loss = loss_fn(outputs,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        batch_size = labels.size(0)

        train_loss_sum += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)
        train_correct +=(predictions == labels).sum().item()
        train_total+=batch_size

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images,labels in val_loader:
            images = images.to(device)
            labels = labels.to(device) 

            outputs = model(images)
            loss = loss_fn(outputs,labels)
            batch_size=labels.size(0)
            val_loss_sum += loss.item() * batch_size

            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += batch_size

    # 整个验证集的平均结果
    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total


    # =========================
    # 3. 保存并打印本轮结果
    # =========================
    baseline_train_loss_list.append(train_loss)
    baseline_train_acc_list.append(train_acc)
    baseline_val_loss_list.append(val_loss)
    baseline_val_acc_list.append(val_acc)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f}"
    )
        

Epoch 01 | train_loss=2.2162 | train_acc=0.3040 | val_loss=2.0712 | val_acc=0.3930
Epoch 02 | train_loss=1.8628 | train_acc=0.4110 | val_loss=1.5830 | val_acc=0.4490
Epoch 03 | train_loss=1.4181 | train_acc=0.5025 | val_loss=1.2734 | val_acc=0.5480
Epoch 04 | train_loss=1.1902 | train_acc=0.5630 | val_loss=1.0487 | val_acc=0.6330
Epoch 05 | train_loss=1.1002 | train_acc=0.5790 | val_loss=1.0073 | val_acc=0.6230
Epoch 06 | train_loss=0.9784 | train_acc=0.6250 | val_loss=0.9323 | val_acc=0.6740
Epoch 07 | train_loss=0.9221 | train_acc=0.6415 | val_loss=0.8440 | val_acc=0.6880
Epoch 08 | train_loss=0.8785 | train_acc=0.6580 | val_loss=0.9100 | val_acc=0.5960
Epoch 09 | train_loss=0.8326 | train_acc=0.6775 | val_loss=0.9142 | val_acc=0.6360
Epoch 10 | train_loss=0.8219 | train_acc=0.6705 | val_loss=0.8429 | val_acc=0.6860
Epoch 11 | train_loss=0.7840 | train_acc=0.6980 | val_loss=0.7538 | val_acc=0.6850
Epoch 12 | train_loss=0.7591 | train_acc=0.7065 | val_loss=0.7061 | val_acc=0.7500
Epoc

In [18]:
set_seed(42)

dropout_model = nn.Sequential(
    nn.Flatten(),

    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(p=0.3),

    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(p=0.3),

    nn.Linear(128, 10)
).to(device)

dropout_optimizer = torch.optim.SGD(
    dropout_model.parameters(),
    lr=0.1
)

print(dropout_model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.3, inplace=False)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ReLU()
  (6): Dropout(p=0.3, inplace=False)
  (7): Linear(in_features=128, out_features=10, bias=True)
)


In [19]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)


In [20]:
num_epochs = 20

dropout_train_loss_list = []
dropout_train_acc_list = []

dropout_val_loss_list = []
dropout_val_acc_list = []


# =========================
# 3. 开始训练
# =========================

for epoch in range(num_epochs):

    # -------------------------
    # 训练阶段
    # -------------------------

    dropout_model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        # 把数据移动到 CPU / GPU
        images = images.to(device)
        labels = labels.to(device)

        # 前向传播
        outputs = dropout_model(images)

        # 计算损失
        loss = loss_fn(outputs, labels)

        # 清空之前的梯度
        dropout_optimizer.zero_grad()

        # 反向传播
        loss.backward()

        # 更新参数
        dropout_optimizer.step()


        # -------------------------
        # 统计训练结果
        # -------------------------

        batch_size = labels.size(0)

        train_loss_sum += loss.item() * batch_size

        predictions = outputs.argmax(dim=1)

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += batch_size


    # 整个训练集的平均结果
    train_loss = train_loss_sum / train_total

    train_acc = train_correct / train_total


    # -------------------------
    # 验证阶段
    # -------------------------

    dropout_model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            # 只做前向传播
            outputs = dropout_model(images)

            loss = loss_fn(outputs, labels)


            # -------------------------
            # 统计验证结果
            # -------------------------

            batch_size = labels.size(0)

            val_loss_sum += loss.item() * batch_size

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += batch_size


    # 整个验证集的平均结果
    val_loss = val_loss_sum / val_total

    val_acc = val_correct / val_total


    # -------------------------
    # 保存这一轮结果
    # -------------------------

    dropout_train_loss_list.append(train_loss)
    dropout_train_acc_list.append(train_acc)

    dropout_val_loss_list.append(val_loss)
    dropout_val_acc_list.append(val_acc)


    # -------------------------
    # 打印
    # -------------------------

    print(
        f"Epoch {epoch + 1:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f}"
    )

Epoch 01 | train_loss=2.2510 | train_acc=0.1585 | val_loss=2.1470 | val_acc=0.3510
Epoch 02 | train_loss=2.0184 | train_acc=0.3135 | val_loss=1.7453 | val_acc=0.4340
Epoch 03 | train_loss=1.6064 | train_acc=0.4165 | val_loss=1.3390 | val_acc=0.5390
Epoch 04 | train_loss=1.3343 | train_acc=0.5130 | val_loss=1.1636 | val_acc=0.5440
Epoch 05 | train_loss=1.1710 | train_acc=0.5675 | val_loss=0.9965 | val_acc=0.6430
Epoch 06 | train_loss=1.0816 | train_acc=0.5840 | val_loss=0.9162 | val_acc=0.6590
Epoch 07 | train_loss=1.0094 | train_acc=0.6215 | val_loss=0.9080 | val_acc=0.6180
Epoch 08 | train_loss=0.9614 | train_acc=0.6355 | val_loss=0.8748 | val_acc=0.6380
Epoch 09 | train_loss=0.9437 | train_acc=0.6320 | val_loss=0.8281 | val_acc=0.6610
Epoch 10 | train_loss=0.8667 | train_acc=0.6650 | val_loss=0.7604 | val_acc=0.7090
Epoch 11 | train_loss=0.8557 | train_acc=0.6820 | val_loss=0.7786 | val_acc=0.6890
Epoch 12 | train_loss=0.8122 | train_acc=0.6950 | val_loss=0.7786 | val_acc=0.6810
Epoc